In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
test = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/SVD-0126/result/SVD_Single_Injection/5/f_Adult_1215u_pos5_neg1_all_35recommendation.csv")

In [ ]:
test.head(50)

In [22]:
from pathlib import Path
import pandas as pd
import re

BASE_DIR = Path("/home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/SVD-0126/result/SVD_Single_Injection/5")

PER_BOOK_FP = BASE_DIR / "figures" / "Classics_explanation" / "per_book_ranking.csv"


def parse_dataset(ds):
    ds = ds.strip()

    if ds.startswith("original"):
        K = int(ds.replace("original", ""))
        return "original", None, K

    m = re.match(r"(\d+)u_(\d+)", ds)
    if m:
        N = int(m.group(1))
        K = int(m.group(2))
        return "inj", N, K

    raise ValueError("Unknown dataset format:", ds)


def rec_file_for(genre, kind, N, K):
    if kind == "original":
        return BASE_DIR / f"ORIGINAL_{K}recommendation.csv"

    return BASE_DIR / f"f_{genre}_{N}u_pos5_neg1_all_{K}recommendation.csv"


# ================== RUN SANITY CHECK ==================

df = pd.read_csv(PER_BOOK_FP)

genre = "Classics"
dataset = "original15"   # change if you want

kind, N, K = parse_dataset(dataset)

# numerator: total classics appearances
total_genre_recs = df[df["dataset"] == dataset]["freq"].sum()

# denominator: real number of users
rec_fp = rec_file_for(genre, kind, N, K)
rec_df = pd.read_csv(rec_fp, low_memory=False)

total_users = rec_df["user_id"].nunique()

avg_per_user = total_genre_recs / total_users

print("Genre:", genre)
print("Dataset:", dataset)
print("Users:", total_users)
print("Sum freq:", total_genre_recs)
print("Avg genre books per user (Top-15):", avg_per_user)


Genre: Classics
Dataset: original15
Users: 53424
Sum freq: 47307
Avg genre books per user (Top-15): 0.8855008984725966


In [24]:
import pandas as pd

FILE = "/home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/SVD-0126/result/SVD_Single_Injection/5/figures/_all_genres/per_book_ranking_all.csv"

OLD_NS = [2,4,6,25,50,100,200,300,350,500]

K = 15
TARGET_GENRE = "classics"

df = pd.read_csv(FILE)

df["genre"] = df["genre"].str.lower()

# -------- get real total users from ORIGINAL --------

orig = df[
    (df["genre"] == TARGET_GENRE) &
    (df["dataset"] == f"original{K}")
]

total_users = orig["freq"].sum() / (
    orig["freq"].sum() / 0.8855008984725966   # reverse sanity known avg
)

# simpler: hardcode since we verified
total_users = 53424

print("Total users:", total_users)

# -------- ORIGINAL --------

orig_sum = orig["freq"].sum()
print("original avg:", orig_sum / total_users)

# -------- OLD INJECTIONS --------

for N in OLD_NS:

    sub = df[
        (df["genre"] == TARGET_GENRE) &
        (df["dataset"] == f"{N}u_{K}")
    ]

    if sub.empty:
        continue

    s = sub["freq"].sum()
    avg = s / total_users

    print(f"{N}u avg:", avg)


Total users: 53424
original avg: 0.8855008984725966
2u avg: 1.353661275831087
4u avg: 1.8849019167415393
6u avg: 2.2089697514225817
25u avg: 4.566561844863732
50u avg: 4.2407719377059
100u avg: 4.253387990416292
200u avg: 3.454121743036837
300u avg: 3.5056903264450434
350u avg: 4.061938454627134
500u avg: 3.6035115303983227


In [31]:
import pandas as pd

FILE = "/home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/SVD-0126/result/SVD_Single_Injection/5/figures/_all_genres/per_book_ranking_all.csv"

OLD_NS = [2,4,6,25,50,100,200,300,350,500]

K_LIST = [15,25,35]

OUT_CSV = "/home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/SVD-0126/result/SVD_Single_Injection/5/figures/_all_genres/avg_all_genres_all_K.csv"

# =====================================================
# LOAD
# =====================================================

df = pd.read_csv(FILE)

df["genre"] = df["genre"].str.lower()

rows = []

# =====================================================
# LOOP OVER K
# =====================================================

for K in K_LIST:

    # ---------------- TOTAL USERS FROM ORIGINAL ----------------

    orig_all = df[df["dataset"] == f"original{K}"]

    if orig_all.empty:
        print(f"⚠️ Missing original{K}")
        continue

    total_users = orig_all["users_n"].max()

    print(f"K={K}  total users={total_users}")

    # ---------------- ALL GENRES ----------------

    for genre in sorted(df["genre"].unique()):

        # ===== ORIGINAL =====

        orig = df[
            (df["genre"] == genre) &
            (df["dataset"] == f"original{K}")
        ]

        orig_sum = orig["freq"].sum()

        orig_avg = orig_sum / total_users

        rows.append({
            "genre": genre,
            "K": K,
            "dataset": "original",
            "avg_books_per_user": orig_avg
        })

        # ===== OLD INJECTIONS =====

        for N in OLD_NS:

            sub = df[
                (df["genre"] == genre) &
                (df["dataset"] == f"{N}u_{K}")
            ]

            if sub.empty:
                continue

            s = sub["freq"].sum()

            avg = s / total_users

            rows.append({
                "genre": genre,
                "K": K,
                "dataset": f"{N}u",
                "avg_books_per_user": avg
            })

# =====================================================
# SAVE
# =====================================================

out_df = pd.DataFrame(rows)

out_df = out_df.sort_values(["genre","K","dataset"])

out_df.to_csv(OUT_CSV, index=False)

print("\n✅ Saved:", OUT_CSV)
print("\nPreview:")
print(out_df.head(30))


K=15  total users=14311
K=25  total users=16100
K=35  total users=17344

✅ Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/SVD-0126/result/SVD_Single_Injection/5/figures/_all_genres/avg_all_genres_all_K.csv

Preview:
     genre   K   dataset  avg_books_per_user
6    adult  15      100u           36.043114
7    adult  15      200u           30.517155
4    adult  15       25u           32.198169
1    adult  15        2u            2.820558
8    adult  15      300u           26.082314
9    adult  15      350u           17.842708
2    adult  15        4u            8.247572
10   adult  15      500u           16.038991
5    adult  15       50u           35.853819
3    adult  15        6u           10.186430
0    adult  15  original            2.247083
139  adult  25      100u           46.406894
140  adult  25      200u           40.326149
137  adult  25       25u           42.171925
134  adult  25        2u            3.965404
141  adult  25      300u           34.355031

In [35]:
from pathlib import Path
from typing import Optional, Dict
import re
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =====================================================
# PATHS
# =====================================================

BASE_DIR = Path(
    "/home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/"
    "SVD-0126/result/SVD_Single_Injection/5"
)

ORIG_DIR = BASE_DIR
SEARCH_ROOT = BASE_DIR

OUT_BASE = BASE_DIR / "figures_0102_all_genres"
OUT_BASE.mkdir(parents=True, exist_ok=True)

K_LIST = [15, 25, 35]
N_LIST = [1215, 2000]

INJECTION_PATTERN = "f_*_{n}u_pos5_neg1_all_{k}recommendation.csv"

# =====================================================
# GENRE NORMALIZATION (ROBUST)
# =====================================================

def clean_basic(s: str) -> str:
    s = str(s).strip().lower()
    s = s.replace("’", "'")
    s = re.sub(r"[_\s]+", " ", s)
    return s

GENRE_MAP = {
    "childrens": "children's",
    "children s": "children's",
    "children": "children's",
    "children's": "children's",
    "children_s": "children's",
    "childrens books": "children's",
    "kids": "children's",
    "kids books": "children's",
}

def norm_genre(s: str) -> str:
    s = clean_basic(s)
    s_no_apos = s.replace("'", "")
    return GENRE_MAP.get(s_no_apos, s)

def slugify(x: str) -> str:
    return re.sub(r"_+","_", re.sub(r"[^A-Za-z0-9]+","_",str(x))).strip("_").lower()

# =====================================================
# IO HELPERS
# =====================================================

def load_rec_csv(fp):
    df = pd.read_csv(fp, low_memory=False)
    if "genres_all" not in df.columns:
        df["genres_all"] = ""
    return df

def has_genre(cell, target):
    if pd.isna(cell):
        return False

    for t in str(cell).split(","):
        if norm_genre(t) == target:
            return True

    return False

def extract_genre_from_filename(p: Path) -> Optional[str]:

    m = re.match(
        r"^f_(.+?)_(\d+)u_pos\d+_neg[^_]+_all_(\d+)recommendation\.csv$",
        p.name,
        flags=re.IGNORECASE
    )

    if not m:
        return None

    raw = m.group(1)

    return norm_genre(raw)

def find_injection(genre, n, k):

    patt = INJECTION_PATTERN.format(n=n, k=k)

    for p in SEARCH_ROOT.glob(patt):

        g = extract_genre_from_filename(p)

        if g == genre:
            return p

    return None

# =====================================================
# DETECT ALL GENRES FROM FILES
# =====================================================

genres = set()

for p in SEARCH_ROOT.glob("f_*recommendation.csv"):
    g = extract_genre_from_filename(p)
    if g:
        genres.add(g)

genres = sorted(genres)

print("Detected genres:")
for g in genres:
    print(" -", g)

# =====================================================
# PLOT FUNCTION
# =====================================================

def plot_genre(genre, data_by_k, out_png):

    ks = sorted(data_by_k.keys())

    groups = ["Original"] + [str(n) for n in N_LIST]

    width = 0.8 / len(groups)
    x = list(range(len(ks)))

    plt.figure(figsize=(8.8,4.4), dpi=160)

    for j,g in enumerate(groups):

        offs = [i + (j-(len(groups)-1)/2)*width for i in x]
        vals = [data_by_k[k].get(g,0) for k in ks]

        plt.bar(
            offs,
            vals,
            width=width,
            label=("n="+g if g!="Original" else "Original")
        )

    plt.xticks(x, [f"Top-{k}" for k in ks])
    plt.ylabel("Avg # of genre books per user")
    plt.title(f"SVD – {genre.capitalize()}")
    plt.legend(fontsize=9, ncol=4)
    plt.tight_layout()

    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png)
    plt.close()

# =====================================================
# MAIN
# =====================================================

def main():

    for genre in genres:

        print("\nProcessing:", genre)

        out_dir = OUT_BASE / f"{slugify(genre)}_explanation"
        out_dir.mkdir(parents=True, exist_ok=True)

        data_by_k: Dict[int, Dict[str,float]] = {k:{} for k in K_LIST}

        txt_out = out_dir / "explanation.txt"

        with open(txt_out, "w") as f:

            f.write(f"{genre}:\n")

            # ---------- ORIGINAL ----------

            for K in K_LIST:

                orig_fp = ORIG_DIR / f"ORIGINAL_{K}recommendation.csv"

                if not orig_fp.exists():
                    continue

                df = load_rec_csv(orig_fp)

                total_users = df["user_id"].nunique()

                mask = df["genres_all"].apply(lambda x: has_genre(x, genre))

                freq = mask.sum()

                avg = freq / total_users if total_users > 0 else 0

                data_by_k[K]["Original"] = avg

                f.write(f"original {K}: avg_per_user: {avg:.6f}\n")

            # ---------- NEW INJECTIONS ----------

            for K in K_LIST:
                for N in N_LIST:

                    fp = find_injection(genre, N, K)

                    if not fp:
                        continue

                    df = load_rec_csv(fp)

                    total_users = df["user_id"].nunique()

                    mask = df["genres_all"].apply(lambda x: has_genre(x, genre))

                    freq = mask.sum()

                    avg = freq / total_users if total_users > 0 else 0

                    data_by_k[K][str(N)] = avg

                    f.write(f"{N}u, {K}, avg_per_user: {avg:.6f}\n")

        # ---------- FIGURE ----------

        fig_out = out_dir / f"{slugify(genre)}__svd.png"

        plot_genre(genre, data_by_k, fig_out)

        print("Saved:", out_dir)

    print("\n✅ ALL GENRES FINISHED SUCCESSFULLY")

# =====================================================

if __name__ == "__main__":
    main()


Detected genres:
 - adult
 - adventure
 - children's
 - classics
 - drama
 - fantasy
 - historical
 - horror
 - mystery
 - nonfiction
 - romance
 - science fiction
 - thriller

Processing: adult
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/SVD-0126/result/SVD_Single_Injection/5/figures_0102_all_genres/adult_explanation

Processing: adventure
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/SVD-0126/result/SVD_Single_Injection/5/figures_0102_all_genres/adventure_explanation

Processing: children's
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/SVD-0126/result/SVD_Single_Injection/5/figures_0102_all_genres/children_s_explanation

Processing: classics
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/SVD-0126/result/SVD_Single_Injection/5/figures_0102_all_genres/classics_explanation

Processing: drama
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/SVD-0126/result/SVD_Single_Injection/5/figures